In [21]:
from pyspark.sql import SparkSession
import os
import json

In [22]:
current_path = os.getcwd()

dir_path = os.path.dirname(os.path.dirname(current_path))

with open(dir_path + '/steam/config.json', 'r') as arquivo:
  config = json.load(arquivo)

In [23]:
jar_path = config['jar_path']

# Crie a sessão Spark com o JAR configurado
spark = SparkSession.builder \
    .appName("GoldStep") \
    .config("spark.jars", jar_path) \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

In [24]:
# Obter o diretório atual
current_path = os.getcwd()

# Verificar o ambiente
if "dev" in current_path:
    env = "dev"
elif "prod" in current_path:
    env = "prod"
else:
    env = "unknown"

print(f"A variável de ambiente é: {env}")

A variável de ambiente é: dev


In [25]:
caminho_pasta = dir_path + "/datalake/steam/user/silver"

df = spark.read.format("parquet").load(caminho_pasta)

In [26]:
df.printSchema()

root
 |-- id: long (nullable = true)
 |-- img: string (nullable = true)
 |-- name: string (nullable = true)
 |-- playtime: float (nullable = true)
 |-- short_description: string (nullable = true)
 |-- website: string (nullable = true)
 |-- recommended: string (nullable = true)
 |-- minimum: string (nullable = true)
 |-- link: string (nullable = true)
 |-- file_date: date (nullable = true)



In [27]:
df.show()

+-------+--------------------+--------------------+--------+--------------------+--------------------+--------------------+--------------------+--------------------+----------+
|     id|                 img|                name|playtime|   short_description|             website|         recommended|             minimum|                link| file_date|
+-------+--------------------+--------------------+--------+--------------------+--------------------+--------------------+--------------------+--------------------+----------+
|    730|https://shared.ak...|    Counter-Strike 2|    1.03|For over two deca...|http://counter-st...|                   -|Minimum:OS: Windo...|https://store.ste...|2024-11-09|
|   4700|https://shared.ak...|Total War: MEDIEV...|   16.65|Spanning the most...|http://www.totalw...|                   -|Minimum System Re...|https://store.ste...|2024-11-09|
|   4760|https://shared.ak...|Rome: Total War™ ...|  766.78|Control and conqu...|http://www.totalw...|             

In [28]:
url = config['url']

properties = {
    "user": config['user'],
    "password": config['password'],
    "driver": config['driver']
}

In [29]:
df.write \
.jdbc(url=url, table="user", mode="overwrite", properties=properties)